In [1]:
pip install textblob

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob

In [9]:
"""
CommBank Social Media - Sentiment Analysis

Loads the Social_Media_Posts sheet, runs an automated sentiment
analysis model (TextBlob, NLP polarity scoring) on the post text,
compares it against the dataset's curated sentiment labels, and
summarizes sentiment by topic, post type, and time.
"""

import pandas as pd
from textblob import TextBlob
# 1. Load data

FILE = "commonwealth_bank_social_media_dataset.xlsx"
df = pd.read_excel(FILE, sheet_name="Social_Media_Posts")


# 2. Run automated sentiment analysis on post_text
#    polarity: -1 (negative) to +1 (positive)

def get_polarity(text):
    return TextBlob(str(text)).sentiment.polarity

def classify(polarity):
    if polarity > 0.15:
        return "Positive"
    elif polarity < -0.15:
        return "Negative"
    else:
        return "Neutral"

df["model_sentiment_score"] = df["post_text"].apply(get_polarity)
df["model_sentiment_label"] = df["model_sentiment_score"].apply(classify)


# 3. Compare model output vs. the dataset's curated labels

df["labels_match"] = df["sentiment_label"] == df["model_sentiment_label"]
agreement_rate = df["labels_match"].mean()

print("Model vs. curated label agreement ")
print(f"Agreement rate: {agreement_rate:.1%}\n")

print("=== Curated sentiment_label distribution ===")
print(df["sentiment_label"].value_counts(), "\n")

print("=== Model sentiment_label distribution ===")
print(df["model_sentiment_label"].value_counts(), "\n")


# 4. Average sentiment by topic (using curated scores)

by_topic = (
    df.groupby("topic")["sentiment_score"]
    .agg(["mean", "count"])
    .sort_values("mean")
    .rename(columns={"mean": "avg_sentiment", "count": "post_count"})
)
print(" Average sentiment by topic ")
print(by_topic, "\n")


# 5. Sentiment by post type (brand vs. reply vs. mention vs. quote)

by_type = df.groupby("post_type")["sentiment_score"].mean().sort_values()
print("Average sentiment by post type ")
print(by_type, "\n")


# 6. Sentiment trend over time (by day)

df["post_date"] = pd.to_datetime(df["post_datetime"]).dt.date
by_date = df.groupby("post_date")["sentiment_score"].mean()
print("=== Average sentiment by date ===")
print(by_date, "\n")


# 7. Negative-sentiment posts that need a response (customer pain points)

pain_points = df[
    (df["sentiment_label"] == "Negative") & (df["response_needed"] == "Yes")
][["post_id", "topic", "customer_intent", "post_text"]]

print("Negative posts needing a response ")
print(pain_points.to_string(index=False), "\n")


# 8. Save enriched dataset

df.to_csv("sentiment_analysis_output.csv", index=False)
print("Saved enriched results to sentiment_analysis_output.csv")


Model vs. curated label agreement 
Agreement rate: 33.3%

=== Curated sentiment_label distribution ===
sentiment_label
Positive    26
Neutral     15
Negative    11
Mixed        8
Name: count, dtype: int64 

=== Model sentiment_label distribution ===
model_sentiment_label
Neutral     34
Positive    21
Negative     5
Name: count, dtype: int64 

 Average sentiment by topic 
                      avg_sentiment  post_count
topic                                          
Branch/ATM                -0.633333           3
Outage/service issue      -0.200000           5
Digital banking            0.021538          13
Customer support           0.050000           1
Sustainability             0.170000           2
Fraud & scams              0.214545          11
Security education         0.352000           5
Card services              0.375000           4
Financial wellbeing        0.387778           9
Business banking           0.446667           3
Community                  0.765000           4 

